#  Data Acquisition

AT2 - Weather Intelligence Services

**Purpose:** Pull hourly historical weather observations for Sydney, NSW from the Open-Meteo
Historical Weather API, aggregate to daily granularity, and save a clean daily dataset that
the target-engineering and feature-engineering notebooks will build on.

**Important constraints from the brief:**
- Location fixed: Sydney, NSW (lat -33.8688, lon 151.2093).
- Only data **before 2026-01-01** may be used for training/validation/test.
- Data from 2026 onwards is reserved as "production" data (not used here).
- We pull a generous history (2015-01-01 to 2025-12-31) so there's enough data for
  lag/rolling features and a proper time-based train/val/test split.

**Data folder convention used in this project:**
- `data/raw/` - untouched hourly data exactly as returned by the API
- `data/interim/` - the daily-aggregated dataset produced here (transformed, but not yet
  final modelling data - targets and features haven't been added yet)
- `data/processed/` - final model-ready datasets (X_train, y_train, etc.), created later
  in the target-engineering / feature-engineering notebooks

In [1]:
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")
INTERIM_DATA_DIR = Path("../data/interim")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

LATITUDE = -33.8688
LONGITUDE = 151.2093
TIMEZONE = "Australia/Sydney"

# Pull well before 2026 so we have enough history for lag features + a proper split.
# Cutoff is strict: nothing on/after 2026-01-01 is used for train/val/test.
START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

# Union of every raw variable needed by BOTH targets (CCI + WHC), see assignment brief.
HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "wind_gusts_10m",
    "cloud_cover",
    "precipitation",
    "snowfall",
]

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

## 1. Fetch hourly data from Open-Meteo

Open-Meteo's `/v1/archive` endpoint is called in yearly chunks (safer than one huge
multi-year request: smaller payloads, easier to retry a single failed year, and avoids
any implicit row limits). Results are cached to `data/raw/` so the notebook is reproducible
without re-hitting the API every run.

In [2]:
def fetch_year(year: int, hourly_vars: list[str]) -> pd.DataFrame:
    """Fetch one calendar year of hourly historical weather for Sydney."""
    start = f"{year}-01-01"
    end = f"{year}-12-31"
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start,
        "end_date": end,
        "hourly": ",".join(hourly_vars),
        "timezone": TIMEZONE,
    }
    resp = requests.get(ARCHIVE_URL, params=params, timeout=60)
    resp.raise_for_status()
    payload = resp.json()

    hourly = payload["hourly"]
    df = pd.DataFrame(hourly)
    df["time"] = pd.to_datetime(df["time"])
    return df


def fetch_all_years(start_date: str, end_date: str, hourly_vars: list[str]) -> pd.DataFrame:
    start_year = pd.Timestamp(start_date).year
    end_year = pd.Timestamp(end_date).year

    frames = []
    for year in range(start_year, end_year + 1):
        cache_path = RAW_DATA_DIR / f"sydney_hourly_{year}.parquet"
        if cache_path.exists():
            df_year = pd.read_parquet(cache_path)
        else:
            df_year = fetch_year(year, hourly_vars)
            df_year.to_parquet(cache_path, index=False)
            time.sleep(1)  # be polite to the free API
        frames.append(df_year)
        print(f"Fetched/loaded {year}: {len(df_year)} hourly rows")

    full = pd.concat(frames, ignore_index=True).sort_values("time").reset_index(drop=True)
    return full


hourly_df = fetch_all_years(START_DATE, END_DATE, HOURLY_VARS)
hourly_df = hourly_df[(hourly_df["time"] >= START_DATE) & (hourly_df["time"] <= f"{END_DATE} 23:59")]
hourly_df.head()

Fetched/loaded 2015: 8760 hourly rows
Fetched/loaded 2016: 8784 hourly rows
Fetched/loaded 2017: 8760 hourly rows
Fetched/loaded 2018: 8760 hourly rows
Fetched/loaded 2019: 8760 hourly rows
Fetched/loaded 2020: 8784 hourly rows
Fetched/loaded 2021: 8760 hourly rows
Fetched/loaded 2022: 8760 hourly rows
Fetched/loaded 2023: 8760 hourly rows
Fetched/loaded 2024: 8784 hourly rows
Fetched/loaded 2025: 8760 hourly rows


,time,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,cloud_cover,precipitation,snowfall
0,2015-01-01 00:00:00,20.8,83,6.0,13.0,5,0.0,0.0
1,2015-01-01 01:00:00,20.6,85,5.3,10.4,4,0.0,0.0
2,2015-01-01 02:00:00,20.5,85,5.6,9.4,1,0.0,0.0
3,2015-01-01 03:00:00,20.3,86,5.7,10.1,0,0.0,0.0
4,2015-01-01 04:00:00,20.2,87,7.1,10.1,2,0.0,0.0


In [3]:
print(hourly_df.shape)
hourly_df.describe()

(96432, 8)


,time,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,cloud_cover,precipitation,snowfall
count,96432,96432.000000,96432.000000,96432.000000,96432.000000,96432.000000,96432.000000,96432.0
mean,2020-07-01 23:30:00,17.622165,73.027252,12.674872,27.269953,48.793305,0.114346,0.0
min,2015-01-01 00:00:00,1.100000,9.000000,0.000000,1.400000,0.000000,0.000000,0.0
25%,2017-10-01 11:45:00,13.800000,62.000000,7.600000,16.900000,5.000000,0.000000,0.0
50%,2020-07-01 23:30:00,17.800000,75.000000,11.500000,25.200000,45.000000,0.000000,0.0
75%,2023-04-02 11:15:00,21.300000,87.000000,16.800000,35.600000,96.000000,0.000000,0.0
max,2025-12-31 23:00:00,42.000000,100.000000,48.800000,102.200000,100.000000,19.700000,0.0
std,NaN,5.258625,16.766277,6.640048,13.259542,39.941889,0.519125,0.0


## 2. Data quality checks

Before aggregating, check for missing hours and obviously invalid values (e.g. negative
precipitation, humidity outside 0-100%). Reanalysis data from Open-Meteo is generally
gap-free, but we verify rather than assume.

In [4]:
# Missingness
print("Missing values per column:")
print(hourly_df.isnull().sum())

# Expected number of hourly rows vs actual (checks for gaps in the time series)
expected_hours = pd.date_range(START_DATE, f"{END_DATE} 23:00", freq="h")
missing_hours = expected_hours.difference(hourly_df["time"])
print(f"\nExpected hours: {len(expected_hours)}, Actual rows: {len(hourly_df)}, Missing timestamps: {len(missing_hours)}")

# Range sanity checks
print("\nRange checks:")
print(f"relative_humidity_2m: [{hourly_df.relative_humidity_2m.min()}, {hourly_df.relative_humidity_2m.max()}]")
print(f"cloud_cover: [{hourly_df.cloud_cover.min()}, {hourly_df.cloud_cover.max()}]")
print(f"precipitation: [{hourly_df.precipitation.min()}, {hourly_df.precipitation.max()}]")
print(f"wind_speed_10m: [{hourly_df.wind_speed_10m.min()}, {hourly_df.wind_speed_10m.max()}]")
print(f"wind_gusts_10m: [{hourly_df.wind_gusts_10m.min()}, {hourly_df.wind_gusts_10m.max()}]")
print(f"snowfall: [{hourly_df.snowfall.min()}, {hourly_df.snowfall.max()}]  (Sydney: expect ~0 always)")

Missing values per column:
time                    0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
wind_gusts_10m          0
cloud_cover             0
precipitation           0
snowfall                0
dtype: int64

Expected hours: 96432, Actual rows: 96432, Missing timestamps: 0

Range checks:
relative_humidity_2m: [9, 100]
cloud_cover: [0, 100]
precipitation: [0.0, 19.7]
wind_speed_10m: [0.0, 48.8]
wind_gusts_10m: [1.4, 102.2]
snowfall: [0.0, 0.0]  (Sydney: expect ~0 always)


## 3. Aggregate to daily granularity

Per the assignment's aggregation table:

| Variable | Aggregation |
|---|---|
| temperature_2m | Daily Mean |
| relative_humidity_2m | Daily Mean |
| wind_speed_10m | Daily Mean |
| cloud_cover | Daily Mean |
| precipitation | Daily Total (sum) |
| wind_gusts_10m | Daily Maximum |
| snowfall | Daily Total (sum) |

In [5]:
df = hourly_df.copy()
df["date"] = df["time"].dt.date

daily_df = df.groupby("date").agg(
    temperature_2m=("temperature_2m", "mean"),
    relative_humidity_2m=("relative_humidity_2m", "mean"),
    wind_speed_10m=("wind_speed_10m", "mean"),
    cloud_cover=("cloud_cover", "mean"),
    precipitation=("precipitation", "sum"),
    wind_gusts_10m=("wind_gusts_10m", "max"),
    snowfall=("snowfall", "sum"),
).reset_index()

daily_df["date"] = pd.to_datetime(daily_df["date"])
daily_df = daily_df.sort_values("date").reset_index(drop=True)

print(daily_df.shape)
daily_df.head()

(4018, 8)


,date,temperature_2m,relative_humidity_2m,wind_speed_10m,cloud_cover,precipitation,wind_gusts_10m,snowfall
0,2015-01-01,23.129167,80.125000,9.466667,26.625000,0.0,37.4,0.0
1,2015-01-02,22.933333,84.958333,9.604167,86.541667,0.0,34.6,0.0
2,2015-01-03,23.754167,85.125000,10.687500,33.833333,0.0,45.4,0.0
3,2015-01-04,24.329167,80.291667,8.329167,52.500000,10.2,46.8,0.0
4,2015-01-05,21.708333,83.166667,10.679167,85.541667,0.5,32.0,0.0


In [6]:
# Sanity check: one row per calendar day, no gaps
expected_days = pd.date_range(START_DATE, END_DATE, freq="D")
missing_days = expected_days.difference(daily_df["date"])
print(f"Expected days: {len(expected_days)}, Actual days: {len(daily_df)}, Missing: {len(missing_days)}")
assert len(missing_days) == 0, "There are missing calendar days - investigate before proceeding"

Expected days: 4018, Actual days: 4018, Missing: 0


## 4. Save the daily aggregated dataset

Saved to `data/interim/` since this is a transformed intermediate dataset - targets
and features haven't been added yet, so it isn't the final modelling data. The
target-engineering and feature-engineering notebooks for both CCI and WHC will load
from here and save their own final model-ready datasets to `data/processed/`.

In [7]:
out_parquet = INTERIM_DATA_DIR / "sydney_daily_weather.parquet"
out_csv = INTERIM_DATA_DIR / "sydney_daily_weather.csv"

daily_df.to_parquet(out_parquet, index=False)
daily_df.to_csv(out_csv, index=False)

print(f"Saved {len(daily_df)} daily rows to:\n  {out_parquet}\n  {out_csv}")

Saved 4018 daily rows to:
  ..\data\interim\sydney_daily_weather.parquet
  ..\data\interim\sydney_daily_weather.csv
